# Extract peptide N-to-C orientation (`n_to_c`) from PDB structures

For each designed peptide, this notebook determines whether the peptide runs
**N-to-C** or **C-to-N** through the binding pocket, using a fixed reference
residue on the target chain as an anchor.

**Definition** (as specified): for a given design's PDB structure, let `resi166`
be the reference residue on the **target chain**. Compare the distance from
that residue to the **peptide chain's N-terminus** vs. its **C-terminus**
(both measured CA-to-CA):

- `n_to_c = 0` if resi166 is closer to the peptide's **C-terminus** than its N-terminus
- `n_to_c = 1` if resi166 is closer to the peptide's **N-terminus** than its C-terminus

**Assumptions (check these before running!):**
- Chain `A` = target/receptor, chain `B` = designed peptide. Flip `TARGET_CHAIN`/
  `PEPTIDE_CHAIN` below if your PDBs are labeled the other way around.
- Residue numbering in the PDB's target chain actually includes a residue
  numbered 166 (i.e. the target's numbering starts near 1, not renumbered per-design).
- Each `Design` value has exactly one matching PDB in `Accepted/`, named
  `{Design}_model1.pdb`, `{Design}_model2.pdb`, etc. -- the first match found
  (lowest model number) is used; a warning is printed if none or several are
  found unexpectedly.

Requires `biopython` (`pip install biopython`).

In [9]:
import os
import glob
import warnings

import numpy as np
import pandas as pd

from Bio.PDB import PDBParser
from Bio.PDB.PDBExceptions import PDBConstructionWarning

warnings.simplefilter("ignore", PDBConstructionWarning)


In [10]:
# ---------------------------------------------------------------------------
# Configuration -- edit per your environment
# ---------------------------------------------------------------------------

DATA_ROOT = "/FastHome/gyula/designs/place_holder/DATA"

# All datasets to process. Add/remove as needed -- this notebook updates each
# dataset's own augmented CSV independently.
DATASETS = ["gy_0", "gy_2", "gy_4", "gy_5", "gy_7", "gy_11_mut_5", "gy_13_maybe_mut_20", "gy_14_more_iter", "j_1", "gy_2nd_iter_1",
                "gy_2nd_iter_try_23_balm_deployed", "gy_2nd_iter_try_24_control_interface_fixing", "gy_2nd_iter_try_29_balm_deployed_weight_10_length_15",  "gy_2nd_iter_try_40_fixed_new_filters", "gy_2nd_iter_try_31_control_length_15",
    "gy_2nd_iter_try_29_balm_deployed_weight_10_length_15",

    "gy_2nd_iter_try_39_fixed_animations",
    "gy_2nd_iter_try_41_more_iter"]

# Subfolders (inside each dataset's directory) to search for PDBs, in order,
# e.g. /FastHome/gyula/designs/place_holder/DATA/gy_0/Accepted/ and .../Rejected/.
# Both are searched for every design; the first match wins (a design should
# only exist in one of them, but this doesn't assume that).
PDB_SUBDIRS = ["Accepted", "Rejected"]

# Chain labeling in the PDBs -- flip these if your files use the opposite convention.
TARGET_CHAIN = "A"
PEPTIDE_CHAIN = "B"

# Reference residue number on the target chain, used as the orientation anchor.
# If TARGET_RESI is missing from a given PDB's chain-A numbering, fall back to
# TARGET_RESI_FALLBACK instead (rather than skipping the design).
TARGET_RESI = 166
TARGET_RESI_FALLBACK = 160

# Column in the augmented CSV that matches PDB filenames (minus the
# "_model<N>.pdb" suffix).
DESIGN_COL = "Design"

OUTPUT_COL = "n_to_c"

# If True, overwrites the original *_augmented.csv in place. If False, writes
# to a new file with a "_with_ntoc" suffix instead, leaving the original
# untouched (safer for a first run / sanity check).
OVERWRITE_IN_PLACE = False


In [11]:
# ---------------------------------------------------------------------------
# PDB lookup + orientation computation
# ---------------------------------------------------------------------------

_pdb_parser = PDBParser(QUIET=True)


def find_pdb_file(pdb_dirs, design_name):
    """Match a Design name to its PDB file, which carries a `_model<N>.pdb`
    suffix the CSV's Design column doesn't have. Searches each directory in
    `pdb_dirs` (e.g. both Accepted/ and Rejected/) in order and returns the
    first match found there (lowest model number within that directory), or
    None if nothing matches in any of them."""
    if isinstance(pdb_dirs, str):
        pdb_dirs = [pdb_dirs]

    for pdb_dir in pdb_dirs:
        pattern = os.path.join(pdb_dir, f"{design_name}_model*.pdb")
        matches = sorted(glob.glob(pattern))
        if matches:
            if len(matches) > 1:
                print(f"    NOTE: {len(matches)} PDB matches for '{design_name}' in "
                      f"{os.path.basename(pdb_dir)}/, using {os.path.basename(matches[0])}")
            return matches[0]
        # fall back to an exact, suffix-less filename in this directory, just in case
        exact = os.path.join(pdb_dir, f"{design_name}.pdb")
        if os.path.exists(exact):
            return exact

    return None


def _get_residue(chain, resi):
    """Standard (non-hetero) residue with sequence number `resi`, or None."""
    for residue in chain:
        het_flag, seqnum, _icode = residue.get_id()
        if het_flag == " " and seqnum == resi:
            return residue
    return None


def _residue_coord(residue):
    """CA coordinate, falling back to the first available atom if CA is missing."""
    if residue is None:
        return None
    if "CA" in residue:
        return residue["CA"].get_coord()
    atoms = list(residue.get_atoms())
    return atoms[0].get_coord() if atoms else None


def _terminal_residues(chain):
    """(N-terminal residue, C-terminal residue) of a chain, by residue number,
    restricted to standard (non-hetero) residues."""
    standard = [r for r in chain if r.get_id()[0] == " "]
    if not standard:
        return None, None
    standard.sort(key=lambda r: r.get_id()[1])
    return standard[0], standard[-1]


def compute_n_to_c(pdb_path, target_chain=TARGET_CHAIN, peptide_chain=PEPTIDE_CHAIN,
                   target_resi=TARGET_RESI, fallback_resi=TARGET_RESI_FALLBACK):
    """Returns 0, 1, or None (if the structure/chains/residue couldn't be resolved).

    Tries `target_resi` (166) first; if that residue number doesn't exist in
    chain A's numbering for this particular PDB, falls back to `fallback_resi`
    (160) instead of giving up on the design entirely.
    """
    structure = _pdb_parser.get_structure("design", pdb_path)
    model = next(structure.get_models())  # first model in the file

    if target_chain not in model or peptide_chain not in model:
        print(f"    WARNING: chain(s) missing in {os.path.basename(pdb_path)} "
              f"(have: {[c.id for c in model]})")
        return None

    target_residue = _get_residue(model[target_chain], target_resi)
    used_resi = target_resi
    if target_residue is None and fallback_resi is not None:
        target_residue = _get_residue(model[target_chain], fallback_resi)
        used_resi = fallback_resi
    if target_residue is None:
        print(f"    WARNING: neither residue {target_resi} nor fallback {fallback_resi} "
              f"found on chain {target_chain} in {os.path.basename(pdb_path)}")
        return None

    n_term_res, c_term_res = _terminal_residues(model[peptide_chain])
    ref_coord = _residue_coord(target_residue)
    n_coord = _residue_coord(n_term_res)
    c_coord = _residue_coord(c_term_res)
    if ref_coord is None or n_coord is None or c_coord is None:
        print(f"    WARNING: missing coordinates in {os.path.basename(pdb_path)}")
        return None

    dist_to_n = float(np.linalg.norm(ref_coord - n_coord))
    dist_to_c = float(np.linalg.norm(ref_coord - c_coord))

    # closer to C-terminus -> 0, closer to N-terminus -> 1
    return 0 if dist_to_c < dist_to_n else 1


In [12]:
# ---------------------------------------------------------------------------
# Process each dataset's augmented CSV
# ---------------------------------------------------------------------------

def process_dataset(dataset):
    csv_path = os.path.join(DATA_ROOT, dataset, "mpnn_design_stats_augmented.csv")
    pdb_dirs = [os.path.join(DATA_ROOT, dataset, subdir) for subdir in PDB_SUBDIRS]

    if not os.path.exists(csv_path):
        print(f"[{dataset}] SKIPPED -- no augmented CSV at {csv_path}")
        return None

    df = pd.read_csv(csv_path)
    if DESIGN_COL not in df.columns:
        print(f"[{dataset}] SKIPPED -- no '{DESIGN_COL}' column in {csv_path}")
        return None

    print(f"[{dataset}] {len(df)} rows, looking up PDBs in {pdb_dirs}")

    n_to_c_values = []
    n_missing_pdb = 0
    n_failed_compute = 0
    for design_name in df[DESIGN_COL].astype(str):
        pdb_path = find_pdb_file(pdb_dirs, design_name)
        if pdb_path is None:
            n_missing_pdb += 1
            n_to_c_values.append(np.nan)
            continue
        value = compute_n_to_c(pdb_path)
        if value is None:
            n_failed_compute += 1
        n_to_c_values.append(value if value is not None else np.nan)

    df[OUTPUT_COL] = n_to_c_values

    n_ok = df[OUTPUT_COL].notna().sum()
    print(f"[{dataset}] {n_ok}/{len(df)} resolved | "
          f"{n_missing_pdb} missing PDB | {n_failed_compute} failed to compute")
    if n_ok > 0:
        print(f"[{dataset}] n_to_c value counts: {df[OUTPUT_COL].value_counts(dropna=False).to_dict()}")

    out_path = csv_path if OVERWRITE_IN_PLACE else csv_path.replace(
        "_augmented.csv", "_augmented_with_ntoc.csv")
    df.to_csv(out_path, index=False)
    print(f"[{dataset}] saved -> {out_path}\n")
    return df


results = {d: process_dataset(d) for d in DATASETS}


[gy_0] 545 rows, looking up PDBs in ['/FastHome/gyula/designs/place_holder/DATA/gy_0/Accepted', '/FastHome/gyula/designs/place_holder/DATA/gy_0/Rejected']


/tmp/ipykernel_3728287/185791761.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[OUTPUT_COL] = n_to_c_values


[gy_0] 545/545 resolved | 0 missing PDB | 0 failed to compute
[gy_0] n_to_c value counts: {0: 343, 1: 202}
[gy_0] saved -> /FastHome/gyula/designs/place_holder/DATA/gy_0/mpnn_design_stats_augmented_with_ntoc.csv

[gy_2] 548 rows, looking up PDBs in ['/FastHome/gyula/designs/place_holder/DATA/gy_2/Accepted', '/FastHome/gyula/designs/place_holder/DATA/gy_2/Rejected']


/tmp/ipykernel_3728287/185791761.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[OUTPUT_COL] = n_to_c_values


[gy_2] 548/548 resolved | 0 missing PDB | 0 failed to compute
[gy_2] n_to_c value counts: {0: 339, 1: 209}
[gy_2] saved -> /FastHome/gyula/designs/place_holder/DATA/gy_2/mpnn_design_stats_augmented_with_ntoc.csv

[gy_4] 471 rows, looking up PDBs in ['/FastHome/gyula/designs/place_holder/DATA/gy_4/Accepted', '/FastHome/gyula/designs/place_holder/DATA/gy_4/Rejected']


...

/tmp/ipykernel_3728287/185791761.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[OUTPUT_COL] = n_to_c_values


[gy_2nd_iter_try_29_balm_deployed_weight_10_length_15] 725/725 resolved | 0 missing PDB | 0 failed to compute
[gy_2nd_iter_try_29_balm_deployed_weight_10_length_15] n_to_c value counts: {0: 513, 1: 212}
[gy_2nd_iter_try_29_balm_deployed_weight_10_length_15] saved -> /FastHome/gyula/designs/place_holder/DATA/gy_2nd_iter_try_29_balm_deployed_weight_10_length_15/mpnn_design_stats_augmented_with_ntoc.csv

[gy_2nd_iter_try_39_fixed_animations] 690 rows, looking up PDBs in ['/FastHome/gyula/designs/place_holder/DATA/gy_2nd_iter_try_39_fixed_animations/Accepted', '/FastHome/gyula/designs/place_holder/DATA/gy_2nd_iter_try_39_fixed_animations/Rejected']


/tmp/ipykernel_3728287/185791761.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[OUTPUT_COL] = n_to_c_values


[gy_2nd_iter_try_39_fixed_animations] 690/690 resolved | 0 missing PDB | 0 failed to compute
[gy_2nd_iter_try_39_fixed_animations] n_to_c value counts: {0: 467, 1: 223}
[gy_2nd_iter_try_39_fixed_animations] saved -> /FastHome/gyula/designs/place_holder/DATA/gy_2nd_iter_try_39_fixed_animations/mpnn_design_stats_augmented_with_ntoc.csv

[gy_2nd_iter_try_41_more_iter] 1774 rows, looking up PDBs in ['/FastHome/gyula/designs/place_holder/DATA/gy_2nd_iter_try_41_more_iter/Accepted', '/FastHome/gyula/designs/place_holder/DATA/gy_2nd_iter_try_41_more_iter/Rejected']
[gy_2nd_iter_try_41_more_iter] 1774/1774 resolved | 0 missing PDB | 0 failed to compute
[gy_2nd_iter_try_41_more_iter] n_to_c value counts: {0: 1254, 1: 520}
[gy_2nd_iter_try_41_more_iter] saved -> /FastHome/gyula/designs/place_holder/DATA/gy_2nd_iter_try_41_more_iter/mpnn_design_stats_augmented_with_ntoc.csv



/tmp/ipykernel_3728287/185791761.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[OUTPUT_COL] = n_to_c_values


In [13]:
# ---------------------------------------------------------------------------
# Summary across all datasets
# ---------------------------------------------------------------------------

summary_rows = []
for dataset, df in results.items():
    if df is None:
        continue
    summary_rows.append({
        "dataset": dataset,
        "n_rows": len(df),
        "n_resolved": int(df[OUTPUT_COL].notna().sum()),
        "n_missing": int(df[OUTPUT_COL].isna().sum()),
        "frac_n_to_c_1": float(df[OUTPUT_COL].mean()) if df[OUTPUT_COL].notna().any() else float("nan"),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


,dataset,n_rows,n_resolved,n_missing,frac_n_to_c_1
0,gy_0,545,545,0,0.370642
1,gy_2,548,548,0,0.381387
2,gy_4,471,471,0,0.354565
3,gy_5,560,560,0,0.432143
4,gy_7,563,103,460,0.339806
5,gy_11_mut_5,575,575,0,0.325217
6,gy_13_maybe_mut_20,537,537,0,0.325885
7,gy_14_more_iter,519,519,0,0.263969
8,j_1,102,0,102,NaN
9,gy_2nd_iter_1,747,747,0,0.337349


In [14]:
results.keys()

dict_keys(['gy_0', 'gy_2', 'gy_4', 'gy_5', 'gy_7', 'gy_11_mut_5', 'gy_13_maybe_mut_20', 'gy_14_more_iter', 'j_1', 'gy_2nd_iter_1', 'gy_2nd_iter_try_23_balm_deployed', 'gy_2nd_iter_try_24_control_interface_fixing', 'gy_2nd_iter_try_29_balm_deployed_weight_10_length_15', 'gy_2nd_iter_try_40_fixed_new_filters', 'gy_2nd_iter_try_31_control_length_15', 'gy_2nd_iter_try_39_fixed_animations', 'gy_2nd_iter_try_41_more_iter'])

In [16]:
results["gy_11_mut_5"]

,Design,Protocol,Length,Seed,Helicity,Target_Hotspot,Sequence,InterfaceResidues,MPNN_score,MPNN_seq_recovery,...,Target,proteina,Target_length,proteina_length,delta_total,dG_binding_IE,dG_binding_IE_err,found_in,Y,n_to_c
0,BAX_1F16_l13_s780333_mpnn1,4stage,13,780333,0.95,74-99,SPVEELINFMISL,"B2,B3,B5,B6,B7,B9,B10,B11,B13",1.32,0.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,BAX_1F16_l13_s780333_mpnn2,4stage,13,780333,0.95,74-99,SPVQELIDFMISL,"B2,B3,B4,B5,B6,B7,B9,B10,B11,B13",1.38,0.20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,BAX_1F16_l13_s780333_mpnn3,4stage,13,780333,0.95,74-99,SPVDELIEFMISL,"B2,B3,B5,B6,B7,B9,B10,B11,B13",1.41,0.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,BAX_1F16_l13_s780333_mpnn4,4stage,13,780333,0.95,74-99,SPVQELINFMISL,"B2,B3,B4,B5,B6,B7,B9,B10,B11,B13",1.42,0.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,BAX_1F16_l13_s780333_mpnn5,4stage,13,780333,0.95,74-99,SPVDELINFMISL,"B2,B3,B5,B6,B7,B9,B10,B11,B13",1.42,0.60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
570,BAX_1F16_l24_s615080_mpnn2,4stage,24,615080,0.95,74-99,NLGDPERMHEFTQELMSVLDSIMS,"B1,B2,B3,B7,B8,B9,B10,B11,B12,B14,B15,B16,B18,...",1.62,0.22,...,QPRGGGPTSSEQIMKTGALLLQGFIQDRAGRMGGEAPELALDPVPQ...,NLGDPERMHEFTQELMSVLDSIMS,160.0,24.0,-67.66,-6.96,9.01,accepted,-67.66,0
571,BAX_1F16_l14_s970428_mpnn1,4stage,14,970428,0.95,74-99,PSVVDEMLKMITSF,"B1,B2,B3,B4,B6,B7,B8,B10,B11,B12,B14",1.27,0.00,...,QPRGGGPTSSEQIMKTGALLLQGFIQDRAGRMGGEAPELALDPVPQ...,PSVVDEMLKMITSF,160.0,14.0,-65.77,-38.92,4.83,accepted,-65.77,0
572,BAX_1F16_l14_s970428_mpnn2,4stage,14,970428,0.95,74-99,PSVVDEMLEMITSF,"B3,B4,B6,B7,B8,B10,B11,B12,B14",1.32,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
573,BAX_1F16_l12_s472610_mpnn1,4stage,12,472610,0.95,74-99,SLVDVMMETVQS,"B1,B2,B3,B5,B6,B7,B9,B10,B11,B12",1.35,0.33,...,QPRGGGPTSSEQIMKTGALLLQGFIQDRAGRMGGEAPELALDPVPQ...,SLVDVMMETVQS,160.0,12.0,-54.67,-20.51,4.41,accepted,-54.67,0
